# rl17_report

> Phase 17 analysis (`RL17_PLAN`): team-size curves per model (`curves`: welfare, pooled share, honest loss, sharing, invalid replies at N 4-32), per-run gains over the warm start (`gains`), H17-1 to H17-5 with the exact tests of `rl16_report` (the training run as the unit) and Holm within each family (`rl17Summaries`), Claude Haiku references from Phases 11, 13 and 14 (`haikuRefs`; full memory, so references only), `writeRL17`, `figRL17` and `syntheticRows17`. Exploratory.

In [ ]:
#| default_exp rl17_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.coop_grid import MAIN_SEEDS
from explRL.fix_stats import holm
from explRL.llm_figures import _save
from explRL.llm_report import NAN, _f
from explRL.rl16_report import hierBoot, permTest, signFlip
from explRL.rl17_grid import BIG_SEEDS, RL17_ARMS, RL17_PLAN, RL17_SEEDS, SIZES

METRICS = ('welfare', 'pooledShare', 'sharePerHeld', 'soloShare', 'invalid')
RUNS = {a: [f'{a}_s{s}' for s in range(1, RL17_SEEDS[a] + 1)] for a in RL17_ARMS}


def boards(rows, model, test, metric):
    "{board seed: value} for a model's cells at level '<model>|<test>' (invalid: per text turn)."
    out = {}
    for r in rows:
        if r['level'] == f'{model}|{test}':
            out[int(_f(r['seed']))] = _f(r['invalid']) / max(_f(r['calls']), 1) if metric == 'invalid' else _f(r[metric])
    return out


def _m(b):
    v = [x for x in b.values() if x == x]
    return float(np.mean(v)) if v else NAN


def honest(rows, model, N):
    "{seed: welfareHonest clean - with N // 4 liars} on the boards both have."
    c, l = boards(rows, model, f'n{N}', 'welfareHonest'), boards(rows, model, f'n{N}liar', 'welfareHonest')
    return {s: c[s] - l[s] for s in c if s in l}


def curves(rows, models):
    "{model: {N: {metric: mean}, honestLoss}} at every team size."
    return {m: {N: dict({k: _m(boards(rows, m, f'n{N}', k)) for k in METRICS}, honestLoss=_m(honest(rows, m, N)))
                for N in SIZES} for m in models}


def gains(rows, run, N, metric='welfare', warm='warm'):
    "{seed: run - warm} on the clean boards at N."
    a, w = boards(rows, run, f'n{N}', metric), boards(rows, warm, f'n{N}', metric)
    return {s: a[s] - w[s] for s in a if s in w}


def haikuRefs(clue, scale, fix):
    "Haiku's welfare and pooled share per N (Phase 11 N 4; Phase 14 N 8-32) and honest loss (Phase 13 N 4; Phase 14)."
    sel = lambda rs, **kw: [r for r in rs if all(str(r.get(k)) == str(v) for k, v in kw.items())]
    mean = lambda rs, k: float(np.mean([_f(r[k]) for r in rs])) if rs else NAN
    out = {}
    for N in SIZES:
        cl = sel(clue, study='clue', runner='llm', channel='blackboard') if N == 4 else []
        cl = [r for r in cl if int(_f(r['N'])) == 4 and int(_f(r['seed'])) in range(900, 905)] or \
             [r for r in sel(scale, study='scaleB', level='baseline') if int(_f(r['N'])) == N]
        src = sel(fix, study='fixA') if N == 4 else sel(scale, study='scaleB')
        c = {int(_f(r['seed'])): _f(r['welfareHonest']) for r in src if r['level'] == 'baseline' and int(_f(r['N'])) == N}
        l = {int(_f(r['seed'])): _f(r['welfareHonest']) for r in src if r['level'] == 'baseline|liar' and int(_f(r['N'])) == N}
        hl = [c[s] - l[s] for s in c if s in l]
        out[N] = dict(welfare=mean(cl, 'welfare'), pooledShare=mean(cl, 'pooledShare'), honestLoss=float(np.mean(hl)) if hl else NAN)
    return out

## Hypotheses

In [ ]:
#| export
def rl17Summaries(rows, s0rows=None, refs=None, nboot=2000):
    """RL17_PLAN: curves; per run the gains; H17-1 to H17-5 (exact tests over runs), Holm within each family,
    hierarchical intervals; the warm start's rows may come from step 0 (s0rows)."""
    rows = list(rows) + [r for r in (s0rows or []) if r['level'].startswith('warm|')]
    runs = [n for a in RUNS for n in RUNS[a]]
    tests = [f'n{N}{x}' for N in SIZES for x in ('', 'liar')]
    missing = [m for m in ['warm'] + runs if any(not boards(rows, m, t, 'welfare') for t in tests)]
    if missing:
        raise ValueError(f'missing evaluation rows for {missing}: need the warm start and every run on the whole grid')
    g = lambda run, N, k='welfare': gains(rows, run, N, k)
    mix, four = RUNS['r17mix'], RUNS['r17four']
    hw = honest(rows, 'warm', 32)
    hl = {r: {s: hw[s] - v for s, v in honest(rows, r, 32).items() if s in hw} for r in mix}
    d24 = {r: {s: g(r, 32)[s] - _m(g(r, 4)) for s in g(r, 32)} for r in mix}      # per run: gain at 32 minus its mean gain at 4
    H = dict(
        H17_1=dict(signFlip([_m(g(r, 4)) for r in mix]), ci=hierBoot({'a': {r: g(r, 4) for r in mix}}, lambda d: d['a'].mean(), nboot)),
        H17_2=dict(signFlip([_m(g(r, 32)) - _m(g(r, 4)) for r in mix]), ci=hierBoot({'a': d24}, lambda d: d['a'].mean(), nboot)),
        H17_3=dict(signFlip([_m(g(r, 32, 'pooledShare')) for r in mix]),
                   ci=hierBoot({'a': {r: g(r, 32, 'pooledShare') for r in mix}}, lambda d: d['a'].mean(), nboot)),
        H17_4=dict(signFlip([_m(hl[r]) for r in mix]), ci=hierBoot({'a': hl}, lambda d: d['a'].mean(), nboot)),
        H17_5=dict(permTest([_m(g(r, 32)) for r in mix], [_m(g(r, 32)) for r in four]),
                   ci=hierBoot({'a': {r: g(r, 32) for r in mix}, 'b': {r: g(r, 32) for r in four}},
                               lambda d: d['a'].mean() - d['b'].mean(), nboot)))
    for fam, keys in RL17_PLAN['families'].items():
        for k, q in zip(keys, holm([H[k]['p'] for k in keys])):
            H[k].update(family=fam, pHolm=float(q), supported=bool(q < RL17_PLAN['alpha']))
    per = {r: dict(gain4=_m(g(r, 4)), gain32=_m(g(r, 32)), pooled32=_m(g(r, 32, 'pooledShare'))) for r in runs}
    return dict(curves=curves(rows, ['warm'] + runs), runs=per, hyp=H, refs=refs or {})

## Report, figure, synthetic rows

In [ ]:
#| export
def _g(x, d=3):
    return 'n/a' if x is None or x != x else f'{x:.{d}f}'


def writeRL17(path, S, meta):
    "report.md: provenance, hypotheses, the curves (warm start, arm means, Haiku references), each run."
    L = [f"# Phase 17: RL that makes larger teams pay off ({meta['mode']})", '', 'Exploratory, pre-specified in `RL17_PLAN`; the '
         'training run is the unit (exact tests); intervals are hierarchical bootstrap and descriptive; Haiku curves are '
         'references (full memory).', '', f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`", '',
         '## Hypotheses', '', '| ID | family | estimate [95% CI] | p (exact) | p (Holm) | supported |', '|---|---|---|---|---|---|']
    for k, v in S['hyp'].items():
        est = v.get('mean', v.get('diff'))
        L.append(f"| {k.replace('_', '-')} | {v['family']} | {_g(est)} [{_g(v['ci'][0])}, {_g(v['ci'][1])}] | {_g(v['p'], 4)} | "
                 f"{_g(v['pHolm'], 4)} | {'yes' if v['supported'] else 'no'} |")
    arm = lambda a, N, k: float(np.nanmean([S['curves'][r][N][k] for r in RUNS[a]]))
    for k in ('welfare', 'pooledShare', 'honestLoss', 'sharePerHeld', 'invalid'):
        L += ['', f'## {k} against team size', '', '| model | ' + ' | '.join(f'N {N}' for N in SIZES) + ' |', '|---|' + '---|' * len(SIZES)]
        L.append('| warm start | ' + ' | '.join(_g(S['curves']['warm'][N][k]) for N in SIZES) + ' |')
        L += [f'| {a} (mean of {len(RUNS[a])}) | ' + ' | '.join(_g(arm(a, N, k)) for N in SIZES) + ' |' for a in RUNS]
        if S['refs'] and k in ('welfare', 'pooledShare', 'honestLoss'):
            L.append('| Claude Haiku (reference) | ' + ' | '.join(_g(S['refs'][N][k]) for N in SIZES) + ' |')
    L += ['', '## Runs', '', '| run | gain N 4 | gain N 32 | pooled share gain N 32 |', '|---|---|---|---|']
    L += [f"| {r} | {_g(v['gain4'])} | {_g(v['gain32'])} | {_g(v['pooled32'])} |" for r, v in S['runs'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def figRL17(out, S):
    "rl17.png: welfare, pooled share and honest loss against team size: warm start, each run (thin) and arm means, Haiku."
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
    cols = dict(r17mix='#1f6391', r17four='#e67e22')
    for ax, (k, t) in zip(axs, (('welfare', 'welfare per agent-turn'), ('pooledShare', 'pooled share (clean)'),
                               ('honestLoss', 'honest loss from liars'))):
        ax.plot(SIZES, [S['curves']['warm'][N][k] for N in SIZES], 'k--', label='imitation (warm start)')
        for a, c in cols.items():
            for r in RUNS[a]:
                ax.plot(SIZES, [S['curves'][r][N][k] for N in SIZES], color=c, alpha=0.25, lw=1)
            ax.plot(SIZES, [np.nanmean([S['curves'][r][N][k] for r in RUNS[a]]) for N in SIZES], color=c, lw=2.5, label=a)
        if S['refs']:
            ax.plot(SIZES, [S['refs'][N][k] for N in SIZES], ':', color='gray', label='Claude Haiku (reference)')
        ax.set(xscale='log', xticks=SIZES, xlabel='team size', title=t)
        ax.set_xticklabels([str(N) for N in SIZES])
    axs[0].legend(fontsize=7)
    return _save(fig, out / 'rl17.png')


def syntheticRows17(seed=0, slope=0.02, base=0.3):
    "Evaluation rows for the warm start and the ten runs: r17mix gains slope x log2(N / 4) + 0.05, r17four 0.05; liars cost 0.2."
    rng, rows = np.random.default_rng(seed), []
    for m in ['warm'] + [r for a in RUNS for r in RUNS[a]]:
        off = rng.normal(0, 0.01)
        for N in SIZES:
            k = np.log2(N / 4)
            g = 0.0 if m == 'warm' else 0.05 + (slope * k if m.startswith('r17mix') else 0.0)
            for test, seeds in ((f'n{N}', MAIN_SEEDS if N == 4 else BIG_SEEDS), (f'n{N}liar', BIG_SEEDS)):
                for sd in seeds:
                    w = base - 0.02 * k + g + off + rng.normal(0, 0.03) - (0.2 - (0.05 if m.startswith('r17mix') else 0)) * test.endswith('liar')
                    rows.append(dict(level=f'{m}|{test}', seed=sd, welfare=w, welfareHonest=w, pooledShare=w + 0.1, sharePerHeld=0.9,
                                     soloShare=0.02, invalid=0.0, calls=100))
    return rows

## Tests

In [ ]:
import tempfile, pathlib
import numpy as np
from explRL.rl17_report import *

rows = syntheticRows17()
S = rl17Summaries(rows, nboot=200)
assert set(S['hyp']) == {f'H17_{i}' for i in range(1, 6)} and set(S['curves']) == {'warm'} | {r for a in RUNS for r in RUNS[a]}
assert all(S['hyp'][k]['supported'] for k in S['hyp']), {k: (v['p'], v['pHolm']) for k, v in S['hyp'].items()}
assert abs(S['hyp']['H17_1']['p'] - 1 / 256) < 1e-12 and abs(S['hyp']['H17_5']['p'] - 1 / 1287) < 1e-12
assert abs(S['hyp']['H17_1']['pHolm'] - 3 / 256) < 1e-12                      # Holm over three equal p in the primary family
# no effect: nothing supported
S0 = rl17Summaries(syntheticRows17(seed=1, slope=0.0, base=0.3), nboot=200)
assert not S0['hyp']['H17_2']['supported'] and not S0['hyp']['H17_5']['supported']
# missing runs are refused; the warm start may come from step 0
try:
    rl17Summaries([r for r in rows if not r['level'].startswith('r17mix_s8|')], nboot=10)
    raise AssertionError
except ValueError:
    pass
try:
    rl17Summaries([r for r in rows if r['level'] != 'r17mix_s1|n32liar'], nboot=10)     # liar boards are required too
    raise AssertionError
except ValueError:
    pass
warm = [r for r in rows if r['level'].startswith('warm|')]
S2 = rl17Summaries([r for r in rows if not r['level'].startswith('warm|')], s0rows=warm, nboot=200)
assert S2['hyp']['H17_1']['p'] == S['hyp']['H17_1']['p']
# curves and honest loss on the synthetic truth (liars cost 0.2 for the warm start)
assert abs(S['curves']['warm'][8]['honestLoss'] - 0.2) < 0.03
with tempfile.TemporaryDirectory() as d:
    refs = {N: dict(welfare=0.5, pooledShare=0.4, honestLoss=0.5) for N in (4, 8, 16, 32)}
    S3 = dict(S, refs=refs)
    p = writeRL17(pathlib.Path(d) / 'report.md', S3, dict(mode='test', planHash='x', commit='y'))
    assert 'H17-2' in p.read_text() and 'Claude Haiku' in p.read_text() and figRL17(pathlib.Path(d), S3).exists()

In [ ]:
# Haiku references from the saved Phase 11, 13 and 14 rows, when present
from explRL.bench_grid import readCSV
R = pathlib.Path('../results/multiagent') if pathlib.Path('../results').exists() else pathlib.Path('results/multiagent')
if (R / 'scale/full/runs.csv').exists():
    ref = haikuRefs(readCSV(R / 'clue/full/runs.csv'), readCSV(R / 'scale/full/runs.csv'), readCSV(R / 'fix/full/runs.csv'))
    assert abs(ref[4]['welfare'] - 0.546) < 0.01 and abs(ref[32]['pooledShare'] - 0.34) < 0.01 and abs(ref[8]['honestLoss'] - 0.47) < 0.01, ref

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()